### Baseline Preditiva: Classificação na Base Integral


In [1]:
from pathlib import Path
import pandas as pd

base_path = Path.cwd().parent
file_path = base_path / "data" / "processed" / "Base_de_Dados_Processada.csv"

df = pd.read_csv(file_path)

df.head()

,id_municipio,População_2022,Pontuação Indice ODS 2025,Goal 1 Score,Goal 2 Score,Goal 3 Score,Goal 5 Score,Goal 6 Score,Goal 7 Score,Goal 8 Score,...,Normalizado (0-100): SDG15_8_UNI_CNS,Normalizado (0-100): SDG15_PRTAMB,Normalizado (0-100): SDG16_1_OB_HM,Normalizado (0-100): SDG16_3_M_ARM_FG,Normalizado (0-100): SDG16_4_TX_HMC,Normalizado (0-100): SDG16_CRP,Normalizado (0-100): SDG16_DRTHMN,Normalizado (0-100): SDG16_TRANSP,Normalizado (0-100): SDG17_1_INVST_PB,Normalizado (0-100): SDG17_3_P_RC_TRB
0,1100015,21495,41.885064,53.769451,53.255278,66.451721,58.730307,20.177142,56.309444,46.940699,...,30.659,80,0.0,0.0,38.784211,85.714,42.857,66.667,0.000000,8.108054
1,1100023,96833,45.475390,54.219225,50.890756,65.677811,41.654565,72.408554,68.007222,49.281375,...,0.013,80,0.0,0.0,32.057895,85.714,57.143,66.667,1.133196,34.298246
2,1100031,5363,47.569518,49.228293,43.817398,89.494428,48.938353,44.025511,63.241944,54.613778,...,0.000,0,0.0,0.0,1.642105,85.714,42.857,66.667,38.715329,8.963317
3,1100049,86895,48.582323,50.752090,51.083977,73.026011,47.023041,66.630268,69.962222,45.581682,...,0.012,80,0.0,0.0,63.655263,85.714,57.143,50.000,12.187507,40.743620
4,1100056,15890,48.718897,56.552994,50.268036,79.958615,32.086768,65.794541,68.762222,52.890151,...,40.055,80,100.0,0.0,33.755263,100.000,42.857,66.667,8.461212,21.905901


Filtro: apenas municípios de Santa Catarina (SC)

Restringe a base a um único estado para reduzir o tamanho e a heterogeneidade dos dados
(municípios de um mesmo estado tendem a ser mais parecidos entre si em rede de ensino,
orçamento, etc.). A UF não está na base processada, então é lida direto da planilha bruta
só para filtrar, e não entra como feature.

In [2]:
raw_path = base_path / "data" / "raw" / "Base_de_Dados_IDSC-BR_2025.xlsx"
uf_por_municipio = pd.read_excel(raw_path, sheet_name="Todos os Dados", usecols=["ID", "UF"])
uf_por_municipio = uf_por_municipio.rename(columns={"ID": "id_municipio"})

ids_sc = uf_por_municipio.loc[uf_por_municipio["UF"] == "SC", "id_municipio"]
df = df[df["id_municipio"].isin(ids_sc)].reset_index(drop=True)

print(f"Municípios de SC na base: {len(df)}")


Municípios de SC na base: 295


Importação das dependências e modelos 

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesRegressor
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# ---------------------------------------------------------------------------
# Alvo de classificação. Escolha UMA das opções em MODO_ALVO:
#   'painel'       -> cores do Painel oficial do IDSC-BR (Vermelho/Laranja/Amarelo/Verde),
#                     definidas pelo próprio IDSC a partir da nota bruta do IDEB
#   'faixas_bruta' -> 5 faixas sobre a NOTA BRUTA do IDEB (x10): Insuficiente (<4,0),
#                     Básico (4,0-4,9), Intermediário (5,0-5,9), Adequado/Meta PNE
#                     (6,0-6,9) e Avançado (>=7,0)
# A nota bruta e o Painel NÃO entram como features: ficam em um arquivo separado
# (Base_de_Alvos_IDEB.csv), gerado pelo notebook 02b_alvos_ideb.ipynb.
# ---------------------------------------------------------------------------
MODO_ALVO = 'painel'  # 'painel' ou 'faixas_bruta'

MAPA_PAINEL = {'red': 'Vermelho', 'orange': 'Laranja', 'yellow': 'Amarelo', 'green': 'Verde'}
ORDEM_PAINEL = ['Vermelho', 'Laranja', 'Amarelo', 'Verde']
ORDEM_FAIXAS = [
    'Insuficiente / Crítico',
    'Básico / Baixo',
    'Intermediário',
    'Adequado (Meta PNE)',
    'Avançado / Alto Desempenho',
]
ORDEM_CLASSES = ORDEM_PAINEL if MODO_ALVO == 'painel' else ORDEM_FAIXAS


def carregar_alvos_ideb():
    """Lê a nota bruta e o Painel do IDEB (AF/AI), indexados por id_municipio."""
    caminho = Path.cwd().parent / "data" / "processed" / "Base_de_Alvos_IDEB.csv"
    return pd.read_csv(caminho, index_col='id_municipio')


def rotular_classes(alvos, ab):
    """Devolve, por id_municipio, o rótulo da classe ('af' = Anos Finais, 'ai' = Anos Iniciais)."""
    if MODO_ALVO == 'painel':
        return alvos[f'painel_{ab}'].map(MAPA_PAINEL)
    nota_x10 = (alvos[f'ideb_{ab}_bruto'] * 10).round(4)
    faixas = pd.cut(nota_x10, bins=[-np.inf, 40, 50, 60, 70, np.inf], labels=ORDEM_FAIXAS, right=False)
    return faixas.astype(object)


# Espaços de busca para o RandomizedSearchCV de cada modelo. Os nomes dos
# parâmetros do Random Forest usam o prefixo 'rf__' porque ele vive dentro
# de um Pipeline (step 'rf'); os demais modelos não estão em Pipeline.
ESPACOS_BUSCA = {
    'Random Forest': {
        'n_estimators': [100, 200, 300],  # removido 500 para reduzir uso de memória
        'max_depth': [None, 5, 10, 15, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2', None],
    },
    'XGBoost': {
        'n_estimators': [100, 200, 300, 500],
        'max_depth': [3, 5, 7, 9],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'subsample': [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
    },
    'LightGBM': {
        'n_estimators': [100, 200, 300, 500],
        'num_leaves': [15, 31, 63, 127],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'subsample': [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
    },
    'CatBoost': {
        'iterations': [200, 400, 600],
        'depth': [4, 6, 8, 10],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'l2_leaf_reg': [1, 3, 5, 7],
    },
}


Separação das variáveis alvo

In [4]:
target_col_ideb_af = 'Normalizado (0-100): SDG4_15_IDEB_AF'
target_col_ideb_ai = 'Normalizado (0-100): SDG4_16_IDEB_AI'

# Alvos de classificação (Painel ou faixas da nota bruta), ligados pelo id_municipio
alvos_ideb = carregar_alvos_ideb()
classe_af, classe_ai = 'CLASSE_AF', 'CLASSE_AI'
df[classe_af] = df['id_municipio'].map(rotular_classes(alvos_ideb, 'af'))
df[classe_ai] = df['id_municipio'].map(rotular_classes(alvos_ideb, 'ai'))

# Colunas que NÃO podem virar features: notas normalizadas dos alvos, as classes e o
# identificador (id_municipio correlaciona com a UF e viraria um proxy geográfico)
targets = df[[target_col_ideb_af, target_col_ideb_ai, classe_af, classe_ai, 'id_municipio']].copy()


Divisão da base de dados para treino e teste e execução dos modelos 

In [5]:
import re
from sklearn.utils.class_weight import compute_sample_weight

def avaliar_modelos_por_alvo(df_base, coluna_alvo, colunas_remover, n_iter=10, usar_balanceamento=True):
    print(f"\n--- Avaliação Baseline: {coluna_alvo} ---")
    
    df_clean = df_base.dropna(subset=[coluna_alvo])
    
    X = df_clean.drop(columns=colunas_remover)

    # Alvo de classificação (Painel ou faixas da nota bruta, conforme MODO_ALVO),
    # codificado como 0..n-1 na ordem de ORDEM_CLASSES
    y = pd.Categorical(df_clean[coluna_alvo], categories=ORDEM_CLASSES).codes
    print(f"Baseline (classe majoritária): {pd.Series(y).value_counts(normalize=True).max():.3f} | MODO_ALVO = {MODO_ALVO}")

    # Limpeza dos nomes das colunas para evitar o LightGBMError
    # Substitui caracteres especiais proibidos por JSON/LightGBM por underline "_"
    X.columns = [re.sub(r'[ \":\{\}\[\],\n\t]', '_', str(col)) for col in X.columns]
    
    # Divisão em Treino e Teste (80/20), estratificada pela faixa de desempenho
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # As faixas de desempenho são desbalanceadas. compute_sample_weight('balanced', ...)
    # dá mais peso, no treino, aos exemplos das classes raras -- isso tende a MELHORAR
    # F1/recall das classes raras e PIORAR a acurácia bruta (o modelo passa a "chutar"
    # menos a classe majoritária, que é justamente a que mais acerta por acaso). Com
    # usar_balanceamento=False, os modelos treinam sem esse ajuste e tendem a maximizar
    # a acurácia bruta -- útil para comparar as duas estratégias lado a lado e decidir
    # qual reportar no TCC (e justificar a escolha com números, não só opinião).
    if usar_balanceamento:
        sample_weight_train = compute_sample_weight('balanced', y_train)
    else:
        sample_weight_train = None

    # Número de dobras da validação cruzada limitado pela classe menos frequente do treino
    min_classe = pd.Series(y_train).value_counts().min()
    n_splits = max(2, min(3, min_classe))
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    # O Random Forest usa um imputador iterativo (bem mais lento que o resto do pipeline).
    # Antes ele era refeito do zero a cada uma das ~30 combinações testadas pelo
    # RandomizedSearchCV, o que deixava a busca extremamente lenta (podia levar horas).
    # Agora ele é ajustado UMA única vez aqui, fora da busca — só o classificador é
    # tunado depois, sobre os dados já imputados. (Pequena concessão metodológica: o
    # imputador "vê" o X_train inteiro de uma vez, em vez de ser recalculado a cada
    # dobra da validação cruzada — na prática o ganho de velocidade compensa muito.)
    imputer_rf = IterativeImputer(
        estimator=ExtraTreesRegressor(n_estimators=10, random_state=42),
        random_state=42
    )
    X_train_rf = pd.DataFrame(imputer_rf.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test_rf = pd.DataFrame(imputer_rf.transform(X_test), columns=X_test.columns, index=X_test.index)

    modelos = {
        'Random Forest': RandomForestClassifier(random_state=42),
        'XGBoost': XGBClassifier(random_state=42),
        'LightGBM': LGBMClassifier(random_state=42, verbose=-1),
        'CatBoost': CatBoostClassifier(random_state=42, verbose=0)
    }

    resultados = []
    for nome, modelo in modelos.items():
        # XGBoost, LightGBM e CatBoost recebem os dados diretamente com NaNs (lidam
        # nativamente com valores faltantes). Random Forest usa a versão já imputada.
        X_tr, X_te = (X_train_rf, X_test_rf) if nome == 'Random Forest' else (X_train, X_test)

        # Busca aleatória de hiperparâmetros, otimizando a Acurácia (métrica principal
        # reportada) via validação cruzada estratificada
        busca = RandomizedSearchCV(
            estimator=modelo,
            param_distributions=ESPACOS_BUSCA[nome],
            n_iter=n_iter,
            scoring='accuracy',  # métrica principal agora é a acurácia
            cv=cv,
            random_state=42,
            n_jobs=1,  # evita explosão de processos paralelos (causava crash por falta de RAM)
            error_score='raise',
        )
        # Com bases pequenas (ex.: um único estado), alguma dobra da validação cruzada
        # pode ficar sem nenhum exemplo de uma classe rara. O XGBoost, em especial, quebra
        # nesse caso (ele exige ver as mesmas classes em toda dobra). Isso é tratado aqui
        # em vez de derrubar a avaliação inteira: o modelo problemático é pulado e os
        # demais continuam normalmente.
        try:
            busca.fit(X_tr, y_train, sample_weight=sample_weight_train)
            melhor_modelo = busca.best_estimator_
            y_pred = melhor_modelo.predict(X_te)

            resultados.append({
                'Modelo': nome,
                'Balanceamento': 'Balanceado' if usar_balanceamento else 'Sem balanceamento',
                'Acurácia': accuracy_score(y_test, y_pred),
                'Precisão (Macro)': precision_score(y_test, y_pred, average='macro', zero_division=0),
                'Recall (Macro)': recall_score(y_test, y_pred, average='macro', zero_division=0),
                'F1-Score (Macro)': f1_score(y_test, y_pred, average='macro'),
            })
        except Exception as e:
            print(f"Aviso: {nome} falhou para '{coluna_alvo}' (provavelmente classe rara ausente "
                  f"em alguma dobra da validação cruzada) — pulando. Erro: {e}")
        
    return pd.DataFrame(resultados)


Exibição dos resultados 

In [6]:

# Roda cada alvo nas duas estratégias (balanceado vs. sem balanceamento) e junta os
# resultados numa tabela só, ordenada por Acurácia -- facilita ver, para cada alvo,
# qual estratégia rendeu a maior acurácia e com qual custo em F1/recall das classes raras.

# Anos Finais
resultados_af = pd.concat([
    avaliar_modelos_por_alvo(df, classe_af, targets, usar_balanceamento=True),
    avaliar_modelos_por_alvo(df, classe_af, targets, usar_balanceamento=False),
], ignore_index=True).sort_values(by='Acurácia', ascending=False)
display(resultados_af)

# Anos Iniciais
resultados_ai = pd.concat([
    avaliar_modelos_por_alvo(df, classe_ai, targets, usar_balanceamento=True),
    avaliar_modelos_por_alvo(df, classe_ai, targets, usar_balanceamento=False),
], ignore_index=True).sort_values(by='Acurácia', ascending=False)
display(resultados_ai)



--- Avaliação Baseline: CLASSE_AF ---
Baseline (classe majoritária): 0.447 | MODO_ALVO = painel

--- Avaliação Baseline: CLASSE_AF ---
Baseline (classe majoritária): 0.447 | MODO_ALVO = painel


,Modelo,Balanceamento,Acurácia,Precisão (Macro),Recall (Macro),F1-Score (Macro)
6,LightGBM,Sem balanceamento,0.612903,0.429739,0.465201,0.444731
4,Random Forest,Sem balanceamento,0.612903,0.438889,0.463370,0.436720
7,CatBoost,Sem balanceamento,0.580645,0.383191,0.441392,0.408654
3,CatBoost,Balanceado,0.548387,0.521164,0.531136,0.490730
0,Random Forest,Balanceado,0.516129,0.388889,0.391941,0.389885
5,XGBoost,Sem balanceamento,0.516129,0.378168,0.390110,0.373737
2,LightGBM,Balanceado,0.483871,0.370370,0.364469,0.356151
1,XGBoost,Balanceado,0.483871,0.359848,0.366300,0.361111



--- Avaliação Baseline: CLASSE_AI ---
Baseline (classe majoritária): 0.626 | MODO_ALVO = painel
Aviso: XGBoost falhou para 'CLASSE_AI' (provavelmente classe rara ausente em alguma dobra da validação cruzada) — pulando. Erro: Invalid classes inferred from unique values of `y`.  Expected: [0 1 2], got [1 2 3]

--- Avaliação Baseline: CLASSE_AI ---
Baseline (classe majoritária): 0.626 | MODO_ALVO = painel
Aviso: XGBoost falhou para 'CLASSE_AI' (provavelmente classe rara ausente em alguma dobra da validação cruzada) — pulando. Erro: Invalid classes inferred from unique values of `y`.  Expected: [0 1 2], got [1 2 3]


,Modelo,Balanceamento,Acurácia,Precisão (Macro),Recall (Macro),F1-Score (Macro)
0,Random Forest,Balanceado,0.672727,0.574603,0.495425,0.515127
3,Random Forest,Sem balanceamento,0.654545,0.465686,0.390196,0.364087
5,CatBoost,Sem balanceamento,0.654545,0.444444,0.427451,0.419831
4,LightGBM,Sem balanceamento,0.654545,0.558140,0.473203,0.491662
2,CatBoost,Balanceado,0.636364,0.569940,0.592157,0.579718
1,LightGBM,Balanceado,0.581818,0.472222,0.471242,0.467390


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

base_path = Path.cwd().parent
raw = pd.read_excel(base_path / "data" / "raw" / "Base_de_Dados_IDSC-BR_2025.xlsx",
                    sheet_name="Todos os Dados", usecols=["ID", "UF"]).rename(columns={"ID": "id_municipio"})
ids_sc = raw.loc[raw["UF"] == "SC", "id_municipio"]

alvos = pd.read_csv(base_path / "data" / "processed" / "Base_de_Alvos_IDEB.csv", index_col="id_municipio")
alvos_sc = alvos[alvos.index.isin(ids_sc)]

ORDEM = ['Insuficiente / Crítico', 'Básico / Baixo', 'Intermediário',
         'Adequado (Meta PNE)', 'Avançado / Alto Desempenho']

linhas, resumo = [], []
for ab, nome in [('af', 'Anos Finais'), ('ai', 'Anos Iniciais')]:
    nota = (alvos_sc[f'ideb_{ab}_bruto'] * 10).round(4)
    painel = alvos_sc[f'painel_{ab}']
    faixa = pd.cut(nota, bins=[-np.inf, 40, 50, 60, 70, np.inf], labels=ORDEM, right=False)

    cont = faixa.value_counts().reindex(ORDEM)          # inclui faixas com 0 município
    removidas = int(cont[cont == 1].sum())               # classes com 1 exemplo (removidas)
    final = cont[cont >= 2]                              # mesma regra do pipeline

    resumo.append({'Alvo': nome, 'Municípios de SC': len(alvos_sc),
                   'Com nota bruta': int(nota.notna().sum()), 'Com Painel': int(painel.notna().sum()),
                   'Com nota mas sem Painel': int((nota.notna() & painel.isna()).sum()),
                   'Removidos (classe com 1 município)': removidas, 'Municípios finais': int(final.sum())})
    for f in ORDEM:
        linhas.append({'Alvo': nome, 'Faixa do IDEB': f, 'Municípios': int(cont[f]),
                       'Situação': 'removida (<2)' if cont[f] == 1 else ('sem município' if cont[f] == 0 else 'mantida'),
                       '% do total final': round(100 * cont[f] / final.sum(), 1) if cont[f] >= 2 else np.nan})

pd.set_option('display.width', 200)
print(pd.DataFrame(resumo).to_string(index=False)); print()
tabela = pd.DataFrame(linhas); print(tabela.to_string(index=False))

         Alvo  Municípios de SC  Com nota bruta  Com Painel  Com nota mas sem Painel  Removidos (classe com 1 município)  Municípios finais
  Anos Finais               295             152         152                        0                                   1                151
Anos Iniciais               295             273         273                        0                                   1                272

         Alvo              Faixa do IDEB  Municípios      Situação  % do total final
  Anos Finais     Insuficiente / Crítico           7       mantida               4.6
  Anos Finais             Básico / Baixo          55       mantida              36.4
  Anos Finais              Intermediário          75       mantida              49.7
  Anos Finais        Adequado (Meta PNE)          14       mantida               9.3
  Anos Finais Avançado / Alto Desempenho           1 removida (<2)               NaN
Anos Iniciais     Insuficiente / Crítico           0 sem município   